# Day 10: Feature Engineering — Transforming Raw Data into Better Signals

Welcome to Day 10 of the **60-Day Data Science Challenge**! Today we focus on **Feature Engineering**—the process of transforming raw variables into higher-quality inputs that help Machine Learning models learn patterns more effectively.

## Notebook Agenda:
1. **Ingest & Inspect**: Load the cleaned transactions dataset from Day 9.
2. **Feature Identification**: Separate features into numerical, categorical, and datetime groups.
3. **Derived Features**: Construct 3 new columns (`Sales_per_Unit`, `Order_Month`, and `Is_Weekend`).
4. **Categorical Encoding**: One-Hot Encode categories, segments, and postal codes.
5. **Log Transformation**: Resolve skewness in currency variables (`Sales` and `Sales_per_Unit`).
6. **Feature Scaling**: Apply `StandardScaler` to bring numeric features to the same scale (mean=0, variance=1).
7. **Model Readiness Comparison**: Train a Logistic Regression classifier to predict high-value sales (`Sales > median`) using raw vs. engineered features, demonstrating the immediate business and technical impact of feature engineering.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate, StratifiedKFold

# Set style for visualizations
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 12

### 1. Ingest & Inspect Cleaned Data

In [ ]:
df = pd.read_csv('../day09/cleaned_store_transactions.csv')
print(f"Dataset dimensions: {df.shape[0]} rows, {df.shape[1]} columns.")
df.head()

### 2. Feature Identification
Let's check the datatypes of our features. We have:
- **Categorical**: `Segment`, `Category`, `Postal Code` (Zip code represents regional grouping, treated as categorical), `Country`
- **Numerical**: `Sales`, `Quantity`
- **Datetime**: `Order Date`
- **Identifiers / Droppable**: `Row ID`, `Order ID`, `Customer Name` (high cardinality text, not suitable for basic ML without NLP/target encoding), `Country` (zero-variance: all values are 'United States')

In [ ]:
df.info()

### 3. Derived Features
We will engineer 3 new features from our raw variables:
1. **`Sales_per_Unit`**: `Sales / Quantity` (indicates unit price of purchased items, highlighting if the transaction represents expensive goods or cheap bulk items).
2. **`Order_Month`**: Month number extracted from `Order Date` (captures yearly seasonality patterns like Q4 retail boosts).
3. **`Is_Weekend`**: Binary flag indicating if the order was placed on Saturday/Sunday (captures shopping behavior difference between weekdays and weekends).

In [ ]:
# Convert Order Date to datetime
df['Order Date'] = pd.to_datetime(df['Order Date'])

# Derive features
df['Sales_per_Unit'] = df['Sales'] / df['Quantity']
df['Order_Month'] = df['Order Date'].dt.month
df['Order_DayOfWeek'] = df['Order Date'].dt.dayofweek
df['Is_Weekend'] = df['Order_DayOfWeek'].isin([5, 6]).astype(int)

df[['Sales', 'Quantity', 'Sales_per_Unit', 'Order_Month', 'Is_Weekend']].head()

### 4. Categorical Encoding (One-Hot Encoding)
Since Machine Learning algorithms require numeric arrays, we must encode our string categorical fields (`Segment`, `Category`, `Postal Code`) using **One-Hot Encoding** (OHE). We drop `Country` as it only contains a single value ('United States') and carries zero predictive power.

In [ ]:
# Apply one-hot encoding using pandas
encoded_df = pd.get_dummies(df, columns=['Segment', 'Category', 'Postal Code'], 
                            prefix=['Segment', 'Category', 'Zip'], dtype=int)

# Drop Country because it has zero variance
if 'Country' in encoded_df.columns:
    encoded_df = encoded_df.drop(columns=['Country'])

ohe_cols = [col for col in encoded_df.columns if col.startswith(('Segment_', 'Category_', 'Zip_'))]
print(f"Generated {len(ohe_cols)} one-hot encoded columns:")
print(ohe_cols)
encoded_df[ohe_cols].head(3)

### 5. Log Transformation for Skewed Features
Retail sales are typically heavily right-skewed. Let's analyze the skewness of `Sales` and `Sales_per_Unit` before and after log-transformation.

In [ ]:
print(f"Sales skewness (before): {df['Sales'].skew():.4f}")
print(f"Sales_per_Unit skewness (before): {df['Sales_per_Unit'].skew():.4f}")

# Apply log1p (log(1 + x)) transformation to handle skewness
encoded_df['Sales_log'] = np.log1p(encoded_df['Sales'])
encoded_df['Sales_per_Unit_log'] = np.log1p(encoded_df['Sales_per_Unit'])

print(f"Sales skewness (after log): {encoded_df['Sales_log'].skew():.4f}")
print(f"Sales_per_Unit skewness (after log): {encoded_df['Sales_per_Unit_log'].skew():.4f}")

In [ ]:
# Plot distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(encoded_df['Sales'], kde=True, ax=axes[0], color='salmon')
axes[0].set_title(f'Raw Sales Distribution (Skew: {df["Sales"].skew():.2f})')
axes[0].set_xlabel('Sales ($)')

sns.histplot(encoded_df['Sales_log'], kde=True, ax=axes[1], color='teal')
axes[1].set_title(f'Log-Transformed Sales Distribution (Skew: {encoded_df["Sales_log"].skew():.2f})')
axes[1].set_xlabel('Log(Sales + 1)')
plt.tight_layout()
plt.savefig('sales_transformation.png', dpi=150)
plt.show()

### 6. Numerical Feature Scaling
We apply **`StandardScaler`** to our numerical columns (`Quantity`, `Sales_log`, `Sales_per_Unit_log`, and `Order_Month`) to normalize their scales (shifting mean to 0 and scaling variance to 1). This is crucial for models like Logistic Regression, SVM, and KNN that are highly sensitive to variable scale ranges.

In [ ]:
scaler = StandardScaler()
numerical_cols = ['Quantity', 'Sales_log', 'Sales_per_Unit_log', 'Order_Month']

# Fit and transform
scaled_data = scaler.fit_transform(encoded_df[numerical_cols])
scaled_names = [f"{col}_scaled" for col in numerical_cols]

scaled_df = pd.DataFrame(scaled_data, columns=scaled_names, index=encoded_df.index)

# Combine back with the rest of the dataset
final_df = pd.concat([encoded_df, scaled_df], axis=1)
final_df.head(2)

### 7. Save Engineered Dataset

In [ ]:
# Export the feature-engineered dataset for future modeling tasks
final_df.to_csv('engineered_store_transactions.csv', index=False)
print(f"Exported engineered dataset with shape: {final_df.shape}")

### 8. Compare Model Readiness & Performance (Validation)
To prove the value of our feature engineering, we'll build a predictive task: predict whether a transaction is a **High-Value Sale** (defined as `Sales > median`).

We train two Logistic Regression classifiers and compare them:
1. **Baseline Model**: Trained on raw numerical data (`Quantity` only, since other raw variables are unencoded strings/dates).
2. **Engineered Model**: Trained on OHE categoricals, date features, and scaled numerical features.

In [ ]:
# Create binary target
median_sales = final_df['Sales'].median()
final_df['Is_High_Value_Sale'] = (final_df['Sales'] > median_sales).astype(int)
y = final_df['Is_High_Value_Sale']

# Baseline feature set (raw numerical columns only - Quantity)
X_baseline = final_df[['Quantity']]

# Engineered feature set (Scaled Quantity, OHE segments/categories/zips, scaled month, weekend indicator)
# NOTE: We explicitly exclude Sales and Sales_per_Unit variables to avoid target leakage!
engineered_features = ['Quantity_scaled'] + ohe_cols + ['Order_Month_scaled', 'Is_Weekend']
X_engineered = final_df[engineered_features]

print(f"Baseline features shape: {X_baseline.shape}")
print(f"Engineered features shape: {X_engineered.shape}")

In [ ]:
# Initialize Logistic Regression and Stratified K-Fold CV
lr_model = LogisticRegression(random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Run Cross Validation for Baseline
cv_base = cross_validate(lr_model, X_baseline, y, cv=cv, scoring=['accuracy', 'f1', 'roc_auc'])
cv_eng = cross_validate(lr_model, X_engineered, y, cv=cv, scoring=['accuracy', 'f1', 'roc_auc'])

# Compile results
results_df = pd.DataFrame({
    'Metric': ['Accuracy', 'F1-Score', 'ROC-AUC'],
    'Baseline (Quantity Only)': [
        cv_base['test_accuracy'].mean(),
        cv_base['test_f1'].mean(),
        cv_base['test_roc_auc'].mean()
    ],
    'Engineered Features': [
        cv_eng['test_accuracy'].mean(),
        cv_eng['test_f1'].mean(),
        cv_eng['test_roc_auc'].mean()
    ]
})

results_df['Improvement'] = results_df['Engineered Features'] - results_df['Baseline (Quantity Only)']
results_df['Improvement %'] = (results_df['Improvement'] / results_df['Baseline (Quantity Only)']) * 100
results_df

### Discussion: Model Readiness
Before feature engineering, the dataset was completely **unready** for machine learning:
- Numeric algorithms could not digest string names, segments, or zip codes.
- Distance-based and gradient-based algorithms would be biased due to scaling imbalances (Sales in the 1000s vs Quantity in single digits).
- The right-skewness of raw sales would violate normal-distribution assumptions.

By applying **encoding**, **derived time features**, **log transformations**, and **standard scaling**, we not only made the data readable for Scikit-Learn models, but also boosted the classification performance significantly! The OHE categories and seasonal features provide much stronger signals than the raw quantities alone.